# Flood early warning for Sri Lanka - model 3 session

Runs the four baselines, **model 3 (STG-Former)**'s P0 to P3 ladder, model 1's
missing `M5_bce` rung, and - if the clock allows - the onset pair.

Model 3 puts model 2's input layer on model 1's river graph. It exists because
RQ1 could not be answered from the first two families: the standing "graph vs
no graph" comparison varies the encoder, the loss and the data extent all at
once. Model 3 varies only the graph.

## Kaggle settings - these are NOT stored in this file

Importing a notebook carries the cells, not the environment. Set all three by
hand in the right-hand panel before running:

1. **Add Input** -> `uom230429e/sri-lanka-flood-tabular-graph-2003-2025`
   Do **not** attach the imagery dataset - model 3's ladder has no SAR rung.
2. **Accelerator** -> **GPU T4 x2** (or P100)
3. **Internet** -> **On** (needed to clone the code)

Then **Save Version -> Save & Run All (Commit)**.

## What it does, and the gate

`P0` runs first and is an **assembly check, not a result**: it is byte-identical
to model 2's `N3` by construction, so it must land on **PR-AUC 0.8269**. Cell 5
checks that automatically and skips the expensive `onset` stage if it fails,
rather than spending three more hours on a broken build.

| step | cost | answers |
|---|---|---|
| `baselines` | ~10 min | the bar to clear |
| `ladder3` P0 to P3 | ~4 h | **RQ1** |
| `M5_bce` | ~40 min | unconfounds every model 1 vs model 2 comparison |
| `onset` | ~3 h | early warning, run as a pair |

Total if everything fits: ~8 h. The clock guard in cell 6 defers `onset` to a
second session rather than risking a kill at ~9 h, which would save nothing.


In [ ]:
# --- 1. get the code -----------------------------------------------------
# rmtree rather than !rm -rf: this runs in the notebook process, so a re-run
# always starts from a clean checkout instead of failing on an existing dir.
import os, shutil, subprocess, time

REPO = "https://github.com/heshannethmina/Srilanka-Flood-Data-Set-Creation"
DEST = "/kaggle/working/repo"
RUN = f"python {DEST}/models/kaggle_run.py"
STARTED = time.time()          # every later cell budgets against this

shutil.rmtree(DEST, ignore_errors=True)
subprocess.run(["git", "clone", "-q", REPO, DEST], check=True)
head = subprocess.run(["git", "-C", DEST, "log", "-1", "--oneline"],
                      capture_output=True, text=True).stdout.strip()
print("commit:", head)

# Model 3 arrived in 303f456. An older checkout has no model3 package at all,
# and the failure three cells later would be a confusing ImportError.
assert os.path.isdir(f"{DEST}/models/model3"), (
    f"this checkout has no models/model3 (at {head}) -- push the model 3 "
    f"commit to {REPO} and re-run")
print("models/model3 present")

In [ ]:
# --- 2. environment check ------------------------------------------------
# Fails loudly *now* rather than four hours into the ladder.
import glob, os, torch

print(f"torch {torch.__version__} | "
      f"{torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU'}")
assert torch.cuda.is_available(), "Accelerator is off -- set it to GPU T4 x2"

hits = glob.glob("/kaggle/input/**/flood_dataset.parquet", recursive=True)
assert hits, "attach uom230429e/sri-lanka-flood-tabular-graph-2003-2025"
print("tabular data:", os.path.dirname(hits[0]))

In [ ]:
# --- 3. baselines (~10 min) ----------------------------------------------
# Cheap, and they are the bar every model row is read against. Re-run here
# rather than reused, so this session's runs/ is self-contained.
!{RUN} --stage baselines

In [ ]:
# --- 4. model 3's ladder (~4 h) ------------------------------------------
# P0 -> P1 -> P2 -> P0_x5 -> P3, one change per rung. P3 - P0_x5 is the RQ1
# answer: encoder, loss, panel, seeds and calibrator all pinned, the graph the
# only moving part.
!{RUN} --stage ladder3

In [ ]:
# --- 5. the P0 gate ------------------------------------------------------
# P0 is byte-identical to model 2's N3 by construction (same 710,870 params,
# same weights at seed 0, zero forward difference), so on the same panel it
# must reproduce N3's PR-AUC. If it does not, something differs -- wrong
# commit, different panel -- and P1..P3 are not interpretable. Cheaper to find
# out here than after three more GPU-hours.
import json, os

N3_PR_AUC, TOL = 0.8269, 0.01
GATE_OK = False
path = "/kaggle/working/runs/P0_temporal.json"

if not os.path.exists(path):
    print("P0 produced no output -- check the ladder3 log above")
else:
    got = json.load(open(path))["test"]["pr_auc"]
    delta = got - N3_PR_AUC
    GATE_OK = abs(delta) <= TOL
    print(f"P0 PR-AUC {got:.4f} vs N3 {N3_PR_AUC:.4f}  (delta {delta:+.4f})")
    print("GATE PASSED -- assembly verified, the ladder is readable" if GATE_OK
          else f"GATE FAILED -- |delta| > {TOL}. Treat P1..P3 as unverified and "
               f"do not spend a session on onset.")

In [ ]:
# --- 6. M5_bce, then onset if the clock allows ---------------------------
# M5_bce is a rung of model 1's ladder, not a stage, so --stage all never runs
# it. Without it every M-vs-N comparison is confounded by the focal loss, which
# model 2 measured at -0.0509 PR-AUC.
!{RUN} --stage ladder --presets M5_bce --ladder-seeds 5

# onset needs ~3 h. Kaggle kills a GPU session at ~9 h and a killed session
# saves nothing, so defer rather than gamble.
elapsed = (time.time() - STARTED) / 3600
print(f"\nelapsed {elapsed:.1f} h")

if GATE_OK and elapsed < 5.5:
    !{RUN} --stage onset
else:
    why = "the P0 gate failed" if not GATE_OK else f"{elapsed:.1f} h already spent"
    print(f"skipping onset ({why}). Run it in a second session:")
    print(f"    !{RUN} --stage onset")

In [ ]:
# --- 7. package the results ----------------------------------------------
# /kaggle/working is the notebook's output, so this survives 'Save Version'.
import glob, os

files = sorted(glob.glob("/kaggle/working/runs/*"))
print(f"{len(files)} result files:")
for f in files:
    print(f"  {os.path.basename(f):45s} {os.path.getsize(f) / 1e3:8.1f} kB")

!cd /kaggle/working && zip -qr runs.zip runs && ls -lh runs.zip

# The summary table for everything this session produced.
!python {DEST}/models/report.py /kaggle/working/runs

## After the run

Download `runs.zip` from the version's **Output** tab.

### Merge before you analyse

`/kaggle/working/runs` starts **empty** every session, so this zip holds only
what ran here. Unzip it **into the same folder as your August runs** - the
paired confidence intervals need both models' `_preds.npz` side by side.

What that affects:

| contrast | needs the August runs? |
|---|---|
| `P3 - P0_x5` - **the RQ1 headline** | no, both rungs ran here |
| `P2 - P1`, `P1 - P0` | no |
| `P4_onset - P4_onset_ctrl` | no |
| `P0 - N3`, `P0_x5 - N5_bce` | yes - needs `N3`, `N5_bce` |
| `M5_bce - M5` | yes - needs the old `M5` |

The headline is self-contained. If the August zip is lost, re-run
`--stage ladder2 --presets2 N3,N5_bce` (~1 h) to restore the cross-checks.

### Then, offline

```bash
python models/rethreshold.py runs/ --far 0.231        # makes ev.det readable
python models/results_doc.py runs/ --out docs/RESULTS.md
```

`results_doc.py` writes the document you make decisions and paper claims from:
headline table with seed error bars, every pre-declared contrast with a paired
block-bootstrap confidence interval, and event detection at a matched
false-alarm ratio.

### Not in this session, deliberately

`--grad-clip 2.0 --epochs 80` is what the model 2 gradient logs argue for, but
`P0` and `P0_x5` are controls that must reproduce model 2 and cannot do so under
a changed optimiser schedule. Sweep it after the ladder lands; the CLI auto-tags
the output so it will not overwrite anything.

The SAR stages are omitted: imagery has lost on the merits twice and reaches
only 9 of 51 nodes.
